# Shared Train/Validation/Test Split for Predictive Models

In [1]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The generated 70/15/15 train, validation and test files are shared inputs for all predictive models. The random split is grouped by calendar date, so all spatial units and time buckets of a day remain in the same split. The checks below verify disjoint dates and report complete-day counts plus zero- and positive-demand coverage for every partition.

In [2]:
DATASETS = (
    *PATHS.gold_1h_demand_hexagons.values(),
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_1h_demand_community_area_unfiltered,
    *PATHS.gold_2h_demand_hexagons.values(),
    PATHS.gold_2h_demand_census_tracts,
    PATHS.gold_2h_demand_community_areas,
    PATHS.gold_2h_demand_community_area_unfiltered,
    *PATHS.gold_4h_demand_hexagons.values(),
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
    PATHS.gold_4h_demand_community_area_unfiltered,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: sample
Model-period filter disabled in sample mode
Inputs: ['GOLD_1H_DEMAND_HEXAGON_7.parquet', 'GOLD_1H_DEMAND_HEXAGON_8.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_2H_DEMAND_HEXAGON_7.parquet', 'GOLD_2H_DEMAND_HEXAGON_8.parquet', 'GOLD_2H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_4H_DEMAND_HEXAGON_7.parquet', 'GOLD_4H_DEMAND_HEXAGON_8.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet']
Output directory: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        # Keep every spatial unit and time bucket from the same calendar day
        # in one split. Unique dates are ordered reproducibly by their hash.
        # Validation and test receive exactly the same number of complete days.
        date_assignment = (
            df_split
            .select(pl.col("datetime_hour").dt.date().alias("_split_date"))
            .unique()
            .with_columns(
                pl.col("_split_date").hash(seed=SEED).alias("_split_order")
            )
            .sort(["_split_order", "_split_date"])
            .collect()
            .with_row_index("_date_rank")
        )
        n_dates = date_assignment.height
        n_holdout_dates = round(n_dates * 0.15)
        n_train_dates = n_dates - 2 * n_holdout_dates
        if n_train_dates <= 0 or n_holdout_dates <= 0:
            raise ValueError(f"Not enough dates for grouped 70/15/15 split: {n_dates}")

        date_assignment = (
            date_assignment
            .with_columns(
                pl.when(pl.col("_date_rank") < n_train_dates)
                .then(pl.lit("train"))
                .when(pl.col("_date_rank") < n_train_dates + n_holdout_dates)
                .then(pl.lit("val"))
                .otherwise(pl.lit("test"))
                .alias("_split")
            )
            .select(["_split_date", "_split"])
        )
        bucketed = (
            df_split
            .with_columns(
                pl.col("datetime_hour").dt.date().alias("_split_date")
            )
            .join(date_assignment.lazy(), on="_split_date", how="inner")
        )
        train = bucketed.filter(pl.col("_split") == "train")
        val = bucketed.filter(pl.col("_split") == "val")
        test = bucketed.filter(pl.col("_split") == "test")
        helper_columns = ["_split_date", "_split"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    split_dates = {
        name: frame.select(
            pl.col("datetime_hour").dt.date().alias("date")
        ).unique().collect()
        for name, frame in {"train": train, "val": val, "test": test}.items()
    }
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = split_dates[left].join(split_dates[right], on="date", how="inner")
        if overlap.height:
            raise ValueError(f"Date leakage between {left} and {right}: {overlap.height} dates")
    date_counts = {name: dates.height for name, dates in split_dates.items()}

    target_stats = {}
    for name, frame in {"train": train, "val": val, "test": test}.items():
        stats = frame.select(
            pl.col(TARGET_COL).is_null().sum().alias("null_targets"),
            (pl.col(TARGET_COL) == 0).sum().alias("zero_demand"),
            (pl.col(TARGET_COL) > 0).sum().alias("positive_demand"),
            pl.col(TARGET_COL).min().alias("min_demand"),
            pl.col(TARGET_COL).mean().alias("mean_demand"),
            pl.col(TARGET_COL).max().alias("max_demand"),
        ).collect().row(0, named=True)
        if stats["null_targets"]:
            raise ValueError(
                f"{dataset_path.name} {name} contains null target values"
            )
        if stats["min_demand"] < 0:
            raise ValueError(
                f"{dataset_path.name} {name} contains negative demand"
            )
        if stats["zero_demand"] == 0 or stats["positive_demand"] == 0:
            raise ValueError(
                f"{dataset_path.name} {name} must contain both zero- and "
                "positive-demand observations"
            )
        target_stats[name] = stats

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {
        "counts": counts,
        "date_counts": date_counts,
        "target_stats": target_stats,
        "paths": output_paths,
    }

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Grouped split dates: {date_counts}")
    print(f"Target coverage: {target_stats}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON_7.parquet: {'total': 54768, 'train': 39120, 'val': 7824, 'test': 7824}, shares={'train': 0.71, 'val': 0.14, 'test': 0.14}
Grouped split dates: {'train': 10, 'val': 2, 'test': 2}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 36928, 'positive_demand': 2192, 'min_demand': 0, 'mean_demand': 2.471651329243354, 'max_demand': 403}, 'val': {'null_targets': 0, 'zero_demand': 7396, 'positive_demand': 428, 'min_demand': 0, 'mean_demand': 2.2078220858895707, 'max_demand': 382}, 'test': {'null_targets': 0, 'zero_demand': 7416, 'positive_demand': 408, 'min_demand': 0, 'mean_demand': 2.075792433537832, 'max_demand': 359}}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_tes

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,date,is_holiday,community_area,weather_station_distance_km,food_drink,landmark,shop,train_station,tmpc,relh,sknt,p01m,vsby,wind_dir_sin,wind_dir_cos,station_observed,weather_imputed,precipitation_missing,weather_rain,weather_snow,weather_fog_mist,weather_thunder,weather_freezing,precipitation_trace,weather_qc_corrected,skyc1_CLR,skyc1_FEW,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,date,i8,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2026-04-21 00:00:00,4,2,0,1.0,6.1232e-17,0.781831,0.62349,0.0,1.0,2026-04-21,0,5,18.733201,132.0,8.0,28.0,6.0,10.277778,43.29,8.75,0.0,10.0,-0.086824,-0.992404,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-04-26 00:00:00,4,7,0,1.0,6.1232e-17,-0.781831,0.62349,0.0,1.0,2026-04-26,0,5,18.733201,132.0,8.0,28.0,6.0,7.361111,85.8075,4.5,0.0,10.0,0.613487,0.481125,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-04-22 12:00:00,4,3,12,1.0,6.1232e-17,0.974928,-0.222521,1.2246e-16,-1.0,2026-04-22,0,5,18.733201,132.0,8.0,28.0,6.0,18.75,61.3575,7.0,0.0,8.0,0.803638,0.179691,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,9,11341,1260.111111,670,2663,34.93,3.881111,1.92,9.83,139.91,15.545556,11.2,29.75,22.06,2.451111,0.0,17.0,0.0,0.0,0.0,0.0,0.5,0.055556,0.0,0.5,165.47,18.385556,11.7,30.0,"""Mobile"""
2026-04-21 00:00:00,4,2,0,1.0,6.1232e-17,0.781831,0.62349,0.0,1.0,2026-04-21,0,6,19.553953,422.0,41.0,68.0,21.0,10.277778,43.29,8.75,0.0,10.0,-0.086824,-0.992404,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,5,4737,947.4,255,1688,36.79,7.358,0.73,15.75,115.56,23.112,5.25,51.56,2.0,0.4,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.2,0.0,1.0,119.06,23.812,7.5,51.56,"""Cash"""
2026-04-26 16:00:00,4,7,16,1.0,6.1232e-17,-0.781831,0.62349,-0.866025,-0.5,2026-04-26,0,6,19.553953,422.0,41.0,68.0,21.0,14.305556,67.455,11.0,0.0,10.0,0.954769,0.256515,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,101,90318,894.237624,178,4122,370.23,3.665644,0.38,18.18,1469.53,14.549802,4.44,49.5,86.19,0.853366,0.0,10.05,0.0,0.0,0.0,0.0,6.0,0.059406,0.0,2.0,1600.72,15.848713,4.94,52.19,"""Mobile"""
2026-04-21 00:00:00,4,2,0,1.0,6.1232e-17,0.781831,0.62349,0.0,1.0,2026-04-21,0,8,15.936408,626.0,65.0,175.0,36.0,10.277778,43.29,8.75,0.0,10.0,-0.086824,-0.992404,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,38,18596,489.368421,60,1755,124.18,3.267895,0.1,17.37,439.07,11.554474,3.25,43.5,43.09,1.133947,0.0,9.28,0.0,0.0,0.0,0.0,5.5,0.144737,0.0,2.0,498.66,13.122632,3.25,51.21,"""Mobile"""
2026-04-17 00:00:00,4,5,0,1.0,6.1232e-17,-0.433884,-0.900969,0.0,1.0,2026-04-17,0,7,17.385429,312.0,99.0,44.0,9.0,9.444444,99.085,3.25,0.0,0.25,0.82643,-0.427011,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,8,5398,674.75,333,1381,43.69,5.46125,1.26,16.12,134.34,16.7925,6.75,40.75,7.69,0.96125,0.0,4.48,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,145.53,18.19125,7.86,44.91,"""Mobile"""
2026-04-19 04:00:00,4,7,4,1.0,6.1232e-17,-0.781831,0.62349,0.866025,0.5,2026-04-19,0,1,22.251282,95.0,11.0,31.0,14.0,3.194444,64.3775,5.75,0.0,10.0,-0.79104,-0.607091,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,6,4525,754.166667,252,1339,30.91,5.151667,0.88,11.1,95.85,15.